# True Out-of-Fold (OOF) Stacking Pipeline with Full Optuna Tuning
This notebook implements a rigorous 2-layer stacking ensemble using LightGBM, XGBoost, and CatBoost.
It includes full Optuna hyperparameter tuning for XGBoost and CatBoost to maximize performance.
All base models are trained on `train_original.csv` to learn the native NaN signals.
The meta-model is Logistic Regression.

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
import optuna
import warnings

warnings.filterwarnings('ignore')

## 1. Load Data & Preprocessing

In [2]:
print("Loading data...")
train = pd.read_csv('../datasets/train_original.csv')
test = pd.read_csv('../datasets/test.csv')

X = train.drop(['id', 'addicted_label'], axis=1)
y = train['addicted_label']
X_test = test.drop(['id'], axis=1)

# Proper Ordinal Mappings
stress_mapping = {'Low': 0, 'Medium': 1, 'High': 2, 'Unknown': -1}
impact_mapping = {'No': 0, 'Yes': 1, 'Unknown': -1}

def apply_mappings(df):
    df_out = df.copy()
    df_out['stress_level'] = df_out['stress_level'].map(stress_mapping).fillna(-1).astype(int)
    df_out['academic_work_impact'] = df_out['academic_work_impact'].map(impact_mapping).fillna(-1).astype(int)
    df_out['gender'] = df_out['gender'].fillna('Unknown').astype('category')
    return df_out

X_preprocessed = apply_mappings(X)
X_test_preprocessed = apply_mappings(X_test)

# Feature Engineering
def add_features(df):
    df_out = df.copy()
    denom_screen = df_out['daily_screen_time_hours'].replace(0, 0.001)
    denom_notif = df_out['notifications_per_day'].replace(0, 0.001)

    df_out['social_media_ratio'] = df_out['social_media_hours'] / denom_screen
    df_out['gaming_ratio'] = df_out['gaming_hours'] / denom_screen
    df_out['work_study_ratio'] = df_out['work_study_hours'] / denom_screen
    df_out['app_opens_per_hour'] = df_out['app_opens_per_day'] / denom_screen
    df_out['notifications_to_opens_ratio'] = df_out['app_opens_per_day'] / denom_notif
    df_out['sleep_deficit'] = 8.0 - df_out['sleep_hours']
    return df_out

X_preprocessed = add_features(X_preprocessed)
X_test_preprocessed = add_features(X_test_preprocessed)

categorical_cols = ['gender']
for col in categorical_cols:
    X_preprocessed[col] = X_preprocessed[col].astype('category')
    X_test_preprocessed[col] = X_test_preprocessed[col].astype('category')

# Constants
N_FOLDS = 5
N_TRIALS = 30 # For Optuna tuning

Loading data...


## 2. Optuna Tuning: XGBoost

In [3]:
def objective_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 800),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'tree_method': 'hist',
        'enable_categorical': True,
        'scale_pos_weight': 2.4,
        'random_state': 42,
        'n_jobs': -1
    }
    
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42) # 3-fold for speed during tuning
    scores = []
    
    for train_idx, val_idx in cv.split(X_preprocessed, y):
        X_tr, y_tr = X_preprocessed.iloc[train_idx], y.iloc[train_idx]
        X_val, y_val = X_preprocessed.iloc[val_idx], y.iloc[val_idx]
        
        model = xgb.XGBClassifier(**params)
        model.fit(X_tr, y_tr)
        preds = model.predict_proba(X_val)[:, 1]
        scores.append(roc_auc_score(y_val, preds))
        
    return np.mean(scores)

print(f"Starting XGBoost Optuna tuning ({N_TRIALS} trials)...")
study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(objective_xgb, n_trials=N_TRIALS, show_progress_bar=True)

print(f"Best XGBoost ROC-AUC: {study_xgb.best_value:.5f}")
xgb_best_params = study_xgb.best_params
xgb_best_params.update({'tree_method': 'hist', 'enable_categorical': True, 'scale_pos_weight': 2.4, 'random_state': 42, 'n_jobs': -1})

[I 2026-08-18 20:15:52,188] A new study created in memory with name: no-name-02594a29-9ebb-4987-9426-a369de4711b1


Starting XGBoost Optuna tuning (30 trials)...


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-08-18 20:18:42,240] Trial 0 finished with value: 0.9629179009372167 and parameters: {'n_estimators': 769, 'learning_rate': 0.06288372959309693, 'max_depth': 10, 'min_child_weight': 3, 'colsample_bytree': 0.7006576280112904, 'subsample': 0.8656977898649606, 'reg_alpha': 2.543156841449651, 'reg_lambda': 0.46780703288846137}. Best is trial 0 with value: 0.9629179009372167.
[I 2026-08-18 20:21:31,059] Trial 1 finished with value: 0.9554437024801455 and parameters: {'n_estimators': 797, 'learning_rate': 0.010291543126997763, 'max_depth': 10, 'min_child_weight': 1, 'colsample_bytree': 0.5555534291147002, 'subsample': 0.8303437013221717, 'reg_alpha': 3.6592256852184715, 'reg_lambda': 0.00013110594612024995}. Best is trial 0 with value: 0.9629179009372167.
[I 2026-08-18 20:23:22,448] Trial 2 finished with value: 0.9581477797334585 and parameters: {'n_estimators': 731, 'learning_rate': 0.012832724967204228, 'max_depth': 10, 'min_child_weight': 7, 'colsample_bytree': 0.7439371536200516, 

## 3. Optuna Tuning: CatBoost

In [4]:
def objective_cat(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 300, 800),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10, log=True),
        'random_strength': trial.suggest_float('random_strength', 1e-3, 10, log=True),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'cat_features': categorical_cols,
        'auto_class_weights': 'Balanced',
        'random_seed': 42,
        'verbose': 100
    }
    
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42) # 3-fold for speed during tuning
    scores = []
    
    for train_idx, val_idx in cv.split(X_preprocessed, y):
        X_tr, y_tr = X_preprocessed.iloc[train_idx], y.iloc[train_idx]
        X_val, y_val = X_preprocessed.iloc[val_idx], y.iloc[val_idx]
        
        model = CatBoostClassifier(**params)
        model.fit(X_tr, y_tr)
        preds = model.predict_proba(X_val)[:, 1]
        scores.append(roc_auc_score(y_val, preds))
        
    return np.mean(scores)

print(f"Starting CatBoost Optuna tuning ({N_TRIALS} trials)...")
study_cat = optuna.create_study(direction='maximize')
study_cat.optimize(objective_cat, n_trials=N_TRIALS, show_progress_bar=True)

print(f"Best CatBoost ROC-AUC: {study_cat.best_value:.5f}")
cat_best_params = study_cat.best_params
cat_best_params.update({'cat_features': categorical_cols, 'auto_class_weights': 'Balanced', 'random_seed': 42, 'verbose': 100})

[I 2026-08-18 20:48:47,395] A new study created in memory with name: no-name-97923cc3-7123-4c43-81c4-8d47302a5e69


Starting CatBoost Optuna tuning (30 trials)...


  0%|          | 0/30 [00:00<?, ?it/s]

0:	learn: 0.6814280	total: 207ms	remaining: 1m 19s
100:	learn: 0.3495401	total: 14s	remaining: 39s
200:	learn: 0.3188952	total: 28.7s	remaining: 26s
300:	learn: 0.3101183	total: 43.2s	remaining: 11.8s
382:	learn: 0.3065234	total: 54.5s	remaining: 0us
0:	learn: 0.6816943	total: 162ms	remaining: 1m 1s
100:	learn: 0.3508360	total: 14.2s	remaining: 39.6s
200:	learn: 0.3199229	total: 28.7s	remaining: 26s
300:	learn: 0.3111487	total: 43.7s	remaining: 11.9s
382:	learn: 0.3072945	total: 55.6s	remaining: 0us
0:	learn: 0.6816894	total: 168ms	remaining: 1m 4s
100:	learn: 0.3509906	total: 14.2s	remaining: 39.6s
200:	learn: 0.3198278	total: 29s	remaining: 26.3s
300:	learn: 0.3112517	total: 44s	remaining: 12s
382:	learn: 0.3076096	total: 55.8s	remaining: 0us
[I 2026-08-18 20:51:34,672] Trial 0 finished with value: 0.9387427058400037 and parameters: {'iterations': 383, 'learning_rate': 0.010427093670136285, 'depth': 9, 'l2_leaf_reg': 1.8945192955378514, 'random_strength': 0.006709239506370841, 'baggi

KeyboardInterrupt: 

In [5]:
import json

# 1. Extract the best parameters from the interrupted study
cat_best_params = study_cat.best_params
cat_best_params.update({'cat_features': categorical_cols, 'auto_class_weights': 'Balanced', 'random_seed': 42, 'verbose': 100})

print(f"Best CatBoost ROC-AUC found before stopping: {study_cat.best_value:.5f}")

# 2. Combine both XGBoost and CatBoost best parameters
tuned_params = {
    'xgboost': xgb_best_params,
    'catboost': cat_best_params
}

# 3. Save them to a JSON file
with open('../datasets/tuned_parameters.json', 'w') as f:
    json.dump(tuned_params, f, indent=4)

print("Tuned parameters successfully saved to '../datasets/tuned_parameters.json'!")

Best CatBoost ROC-AUC found before stopping: 0.96248
Tuned parameters successfully saved to '../datasets/tuned_parameters.json'!


## 4. LightGBM Optimal Parameters (Already Tuned)

In [6]:
lgbm_best_params = {
    'n_estimators': 909, 
    'learning_rate': 0.081595, 
    'num_leaves': 66, 
    'max_depth': 9, 
    'min_child_samples': 49, 
    'colsample_bytree': 0.526310, 
    'subsample': 0.778710, 
    'subsample_freq': 5, 
    'reg_alpha': 9.75536, 
    'reg_lambda': 0.001679, 
    'min_split_gain': 0.39105,
    'is_unbalance': True,
    'random_state': 42,
    'verbose': -1,
    'n_jobs': -1
}

# saving the params found

In [7]:
import json

# Combine both XGBoost and CatBoost best parameters
tuned_params = {
    'xgboost': xgb_best_params,
    'catboost': cat_best_params
}

# Save them to a JSON file
with open('tuned_parameters.json', 'w') as f:
    json.dump(tuned_params, f, indent=4)

print("Tuned parameters successfully saved to 'tuned_parameters.json'!")


Tuned parameters successfully saved to 'tuned_parameters.json'!


## 5. Generate Out-of-Fold (OOF) Predictions

In [8]:
cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

# Matrices to store OOF predictions for Layer 1
oof_train_lgb = np.zeros(len(X_preprocessed))
oof_train_xgb = np.zeros(len(X_preprocessed))
oof_train_cat = np.zeros(len(X_preprocessed))

# Matrices to store test predictions across folds
test_preds_lgb = np.zeros(len(X_test_preprocessed))
test_preds_xgb = np.zeros(len(X_test_preprocessed))
test_preds_cat = np.zeros(len(X_test_preprocessed))

print(f"Starting {N_FOLDS}-Fold CV OOF generation with best parameters...")

for fold, (train_idx, val_idx) in enumerate(cv.split(X_preprocessed, y)):
    print(f"--- Fold {fold + 1}/{N_FOLDS} ---")
    
    X_tr, y_tr = X_preprocessed.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X_preprocessed.iloc[val_idx], y.iloc[val_idx]
    
    # 1. LightGBM
    print("  Training LightGBM...")
    model_lgb = lgb.LGBMClassifier(**lgbm_best_params)
    model_lgb.fit(X_tr, y_tr)
    oof_train_lgb[val_idx] = model_lgb.predict_proba(X_val)[:, 1]
    test_preds_lgb += model_lgb.predict_proba(X_test_preprocessed)[:, 1] / N_FOLDS
    print(f"    LGBM Fold {fold + 1} AUC: {roc_auc_score(y_val, oof_train_lgb[val_idx]):.5f}")
    
    # 2. XGBoost
    print("  Training XGBoost...")
    model_xgb = xgb.XGBClassifier(**xgb_best_params)
    model_xgb.fit(X_tr, y_tr)
    oof_train_xgb[val_idx] = model_xgb.predict_proba(X_val)[:, 1]
    test_preds_xgb += model_xgb.predict_proba(X_test_preprocessed)[:, 1] / N_FOLDS
    print(f"    XGB Fold {fold + 1} AUC: {roc_auc_score(y_val, oof_train_xgb[val_idx]):.5f}")
    
    # 3. CatBoost
    print("  Training CatBoost...")
    model_cat = CatBoostClassifier(**cat_best_params)
    model_cat.fit(X_tr, y_tr)
    oof_train_cat[val_idx] = model_cat.predict_proba(X_val)[:, 1]
    test_preds_cat += model_cat.predict_proba(X_test_preprocessed)[:, 1] / N_FOLDS
    print(f"    CatBoost Fold {fold + 1} AUC: {roc_auc_score(y_val, oof_train_cat[val_idx]):.5f}")

Starting 5-Fold CV OOF generation with best parameters...
--- Fold 1/5 ---
  Training LightGBM...
    LGBM Fold 1 AUC: 0.96342
  Training XGBoost...
    XGB Fold 1 AUC: 0.96344
  Training CatBoost...
0:	learn: 0.5728249	total: 254ms	remaining: 3m 19s
100:	learn: 0.2802049	total: 23s	remaining: 2m 36s
200:	learn: 0.2623277	total: 45.7s	remaining: 2m 13s
300:	learn: 0.2521865	total: 1m 8s	remaining: 1m 50s
400:	learn: 0.2454623	total: 1m 27s	remaining: 1m 24s
500:	learn: 0.2402902	total: 1m 44s	remaining: 59.4s
600:	learn: 0.2360288	total: 2m 9s	remaining: 39.9s
700:	learn: 0.2321405	total: 2m 33s	remaining: 18.6s
785:	learn: 0.2292485	total: 2m 54s	remaining: 0us
    CatBoost Fold 1 AUC: 0.96208
--- Fold 2/5 ---
  Training LightGBM...
    LGBM Fold 2 AUC: 0.96404
  Training XGBoost...
    XGB Fold 2 AUC: 0.96407
  Training CatBoost...
0:	learn: 0.5620741	total: 210ms	remaining: 2m 45s
100:	learn: 0.2805508	total: 16s	remaining: 1m 48s
200:	learn: 0.2617574	total: 31.2s	remaining: 1m 30s

## 6. Evaluate Base Models

In [9]:
print("\nBase Models Final OOF ROC-AUC:")
print(f"LGBM OOF AUC:     {roc_auc_score(y, oof_train_lgb):.5f}")
print(f"XGBoost OOF AUC:  {roc_auc_score(y, oof_train_xgb):.5f}")
print(f"CatBoost OOF AUC: {roc_auc_score(y, oof_train_cat):.5f}")


Base Models Final OOF ROC-AUC:
LGBM OOF AUC:     0.96407
XGBoost OOF AUC:  0.96417
CatBoost OOF AUC: 0.96277


## 7. Layer 2: Meta-Model Training (Stacking)

In [10]:
X_train_meta = pd.DataFrame({'lgb': oof_train_lgb, 'xgb': oof_train_xgb, 'cat': oof_train_cat})
X_test_meta = pd.DataFrame({'lgb': test_preds_lgb, 'xgb': test_preds_xgb, 'cat': test_preds_cat})

print("\nTraining Meta-Model (Logistic Regression)...")
meta_model = LogisticRegression(random_state=42)
meta_model.fit(X_train_meta, y)

# Evaluate Meta-Model using CV
meta_oof_preds = np.zeros(len(X_train_meta))
for train_idx, val_idx in cv.split(X_train_meta, y):
    meta_cv_model = LogisticRegression(random_state=42)
    meta_cv_model.fit(X_train_meta.iloc[train_idx], y.iloc[train_idx])
    meta_oof_preds[val_idx] = meta_cv_model.predict_proba(X_train_meta.iloc[val_idx])[:, 1]

print(f"Final Stacking OOF AUC: {roc_auc_score(y, meta_oof_preds):.6f}")

print("\nMeta-Model Weights:")
for model_name, weight in zip(X_train_meta.columns, meta_model.coef_[0]):
    print(f"{model_name.upper():>8s}: {weight:.4f}")


Training Meta-Model (Logistic Regression)...
Final Stacking OOF AUC: 0.964362

Meta-Model Weights:
     LGB: 2.8196
     XGB: 1.9879
     CAT: 2.6010


## 8. Predict and Save Final Submission

In [11]:
print("\nPredicting final test results...")
final_preds = meta_model.predict_proba(X_test_meta)[:, 1]

os.makedirs('../submissions', exist_ok=True)
submission = pd.DataFrame({'id': test['id'], 'addicted_label': final_preds})
submission.to_csv('../submissions/oof_stacking_optuna_submission.csv', index=False)
print("Submission saved to submissions/oof_stacking_optuna_submission.csv")


Predicting final test results...
Submission saved to submissions/oof_stacking_optuna_submission.csv
